# Exercises XP: Text Preprocessing, NER, POS, and Word2Vec

Completed Colab-ready notebook for the guided NLP exercise.

## Setup

Run this cell first. In Google Colab, it installs/updates the required packages and downloads the NLTK and spaCy resources.

In [ ]:
%pip install --quiet spacy nltk gensim matplotlib seaborn pandas scikit-learn scipy==1.12.0 --upgrade

import nltk
import spacy
from spacy.cli import download as spacy_download

resources = [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "tagsets",
]

for resource in resources:
    nltk.download(resource, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("spaCy pipeline:", nlp.pipe_names)

## Exercise 1: Exploring Text Preprocessing, NER, and POS Tags

In [ ]:
data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}

raw_reviews = data['Review']
raw_reviews

### 1.1 Build `preprocess_text()`

The function below lowercases, tokenizes, removes punctuation, removes stopwords, lemmatizes, and returns a cleaned string.

In [ ]:
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
punctuation = set(string.punctuation)


def preprocess_text(text: str) -> str:
    """Lowercase, tokenize, strip punctuation, drop stopwords, and lemmatize a review."""
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token not in punctuation]
    tokens = [token for token in tokens if token.isalpha()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)


for review in raw_reviews:
    print(preprocess_text(review))

### 1.2 Create a cleaned dataset

Keep the original/raw reviews and the cleaned reviews side by side.

In [ ]:
import pandas as pd

raw_df = pd.DataFrame(data)
cleaned_reviews = [preprocess_text(review) for review in raw_reviews]

cleaned_df = pd.DataFrame({
    "Raw_Review": raw_reviews,
    "Cleaned_Review": cleaned_reviews,
})

display(cleaned_df)

for raw, cleaned in zip(raw_reviews, cleaned_reviews):
    print(f"RAW: {raw}")
    print(f"CLEANED: {cleaned}\n")

### 1.3 Named Entity Recognition

`perform_ner()` uses spaCy's `en_core_web_sm` model and returns `(entity text, entity label)` pairs.

In [ ]:
def perform_ner(text: str):
    """Return (entity, label) pairs found by spaCy."""
    doc = nlp(text)
    return [(entity.text, entity.label_) for entity in doc.ents]


for review in raw_reviews[:3]:
    print(review)
    print(perform_ner(review), "\n")

### 1.4 Part-of-Speech Tagging

`perform_pos_tagging()` tokenizes text with NLTK and applies `nltk.pos_tag()`.

In [ ]:
from nltk import pos_tag


def perform_pos_tagging(text: str):
    """Return POS tags for a given text."""
    tokens = word_tokenize(text)
    return pos_tag(tokens)


print(perform_pos_tagging(raw_reviews[0]))
print(perform_pos_tagging(cleaned_reviews[0]))

# Optional: inspect a tag meaning
nltk.help.upenn_tagset("NN")

### 1.5 Apply NER and POS on raw vs cleaned text

This comparison shows how preprocessing affects downstream NLP tasks.

In [ ]:
comparison_rows = []

for raw, cleaned in zip(raw_reviews, cleaned_reviews):
    comparison_rows.append({
        "Raw_Review": raw,
        "Cleaned_Review": cleaned,
        "NER_Raw": perform_ner(raw),
        "NER_Cleaned": perform_ner(cleaned),
        "POS_Raw": perform_pos_tagging(raw),
        "POS_Cleaned": perform_pos_tagging(cleaned),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

for index, row in comparison_df.iterrows():
    print(f"Review {index + 1}")
    print("Raw NER:", row["NER_Raw"])
    print("Cleaned NER:", row["NER_Cleaned"])
    print("Raw POS:", row["POS_Raw"])
    print("Cleaned POS:", row["POS_Cleaned"])
    print("-" * 80)

#### Exercise 1 Analysis

NER usually works better on raw text because capitalization, punctuation, and full sentence structure help spaCy detect proper names such as `McDonald's`, `Thailand`, `Google`, `Bella Italia`, and `Pizza Hut`. After preprocessing, words are lowercased and some context is removed, so NER may miss entities or assign fewer labels.

POS tagging also changes after preprocessing. Raw text includes function words such as determiners, pronouns, auxiliary verbs, and punctuation. Cleaned text mostly keeps content words, so the output is shorter and more focused on nouns, adjectives, and verbs.

## Exercise 2: Plotting the Word Embeddings

### 2.1 Train a Word2Vec model

The Word2Vec model is trained on the preprocessed and tokenized reviews.

In [ ]:
from gensim.models import Word2Vec

tokenized_reviews = [review.split() for review in cleaned_reviews]

w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    epochs=200,
    seed=42,
)

w2v_model

### 2.2 Inspect embedding dimensions

In [ ]:
vector_dimensions = w2v_model.vector_size
vocabulary_size = len(w2v_model.wv)

print("Vector dimensions:", vector_dimensions)
print("Vocabulary size:", vocabulary_size)
print("Embedding matrix shape:", w2v_model.wv.vectors.shape)

first_word = w2v_model.wv.index_to_key[0]
print(f"\nExample vector for '{first_word}':")
print(w2v_model.wv[first_word])

#### Word2Vec Dimension Analysis

The vector dimension is `50` because the model was trained with `vector_size=50`. This means every word in the vocabulary is represented by 50 numeric values. These values are learned from word co-occurrence patterns in the reviews. In larger datasets, words that appear in similar contexts tend to receive similar vectors.

### 2.3 Plot word embeddings

This plot uses the first two vector dimensions as x/y coordinates and labels each point with its word.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_word_embeddings(model, words=None):
    """Scatter-plot the first two Word2Vec dimensions and annotate each word."""
    if words is None:
        words = model.wv.index_to_key
    else:
        words = [word for word in words if word in model.wv]

    x_values = [model.wv[word][0] for word in words]
    y_values = [model.wv[word][1] for word in words]

    plt.figure(figsize=(14, 10))
    plt.scatter(x_values, y_values, alpha=0.75)

    for word, x, y in zip(words, x_values, y_values):
        plt.annotate(word, xy=(x, y), xytext=(4, 3), textcoords="offset points", fontsize=9)

    plt.title("Word2Vec Word Embeddings: First Two Dimensions")
    plt.xlabel("Embedding dimension 1")
    plt.ylabel("Embedding dimension 2")
    plt.grid(True, alpha=0.3)
    plt.show()


plot_word_embeddings(w2v_model)

#### Word Embedding Plot Analysis

Some related words may appear near each other, but the clusters will probably not be very reliable. The dataset has only 10 short reviews, so Word2Vec has very little context to learn from. The plot also uses only the first two dimensions out of 50, which loses most of the information in the vectors.

Possible reasons for weak clustering:

- The corpus is very small.
- Many words appear only once.
- Restaurant names and food words do not repeat enough for strong patterns.
- Plotting only two raw dimensions is a limited visualization method.
- Better results usually require more text, more repeated vocabulary, parameter tuning, or dimensionality reduction such as PCA/t-SNE.

### 2.4 Optional Improvement: PCA Visualization

The assignment asks for a scatter plot of embedding dimensions. The plot above does that. The PCA plot below is optional and often gives a more useful 2D view because it compresses information from all 50 dimensions.

In [ ]:
from sklearn.decomposition import PCA


def plot_word_embeddings_pca(model, words=None):
    """Project Word2Vec vectors to 2D with PCA and plot them."""
    if words is None:
        words = model.wv.index_to_key
    else:
        words = [word for word in words if word in model.wv]

    vectors = np.array([model.wv[word] for word in words])
    points = PCA(n_components=2, random_state=42).fit_transform(vectors)

    plt.figure(figsize=(14, 10))
    plt.scatter(points[:, 0], points[:, 1], alpha=0.75)

    for word, (x, y) in zip(words, points):
        plt.annotate(word, xy=(x, y), xytext=(4, 3), textcoords="offset points", fontsize=9)

    plt.title("Word2Vec Word Embeddings Projected with PCA")
    plt.xlabel("Principal component 1")
    plt.ylabel("Principal component 2")
    plt.grid(True, alpha=0.3)
    plt.show()


plot_word_embeddings_pca(w2v_model)